# WS 12.2 Demo: Watching a Network Learn

In WS 12.2 you compared two networks: a **quick** one (2 training passes) and a **full** one (20 training passes). The full one was much better. But we only looked at the two endpoints — start and finish.

This demo opens up the middle. You'll watch the *same* network improve one training pass at a time, and you'll see two concrete signs that it's "learning":

1. Its accuracy (and precision and recall) climb pass by pass — and the climb looks different for easy digits vs. hard ones.
2. Its internal **weights** — the numbers that make up the network — start as random noise and settle into patterns that look a little like pieces of digits.

This is all the same network architecture as WS 12.2: two hidden layers of 16 neurons each. Nothing new, just a slower camera.

### Setup

In [ ]:
#@title Setup — run this cell
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

def show_digit(image_row, label=None, predicted=None):
    """Display one MNIST digit. image_row is a length-784 array."""
    plt.imshow(image_row.reshape(28, 28), cmap="gray_r")
    plt.axis("off")
    title = ""
    if label is not None:
        title += f"actual: {label}  "
    if predicted is not None:
        title += f"predicted: {predicted}"
    if title:
        plt.title(title)
    plt.show()

In [ ]:
#@title Load MNIST — run this cell (takes ~30 seconds the first time)
mnist = fetch_openml("mnist_784", version=1, as_frame=False, parser="liac-arff")
X_all = mnist.data / 255.0
y_all = mnist.target.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, train_size=10000, test_size=2000, random_state=0
)
print("Training images:", X_train.shape[0])
print("Test images:    ", X_test.shape[0])

---

## Part 1: Training, one pass at a time

Normally we'd call `nn.fit(X, y)` and let the network do 20 training passes in one shot. But to watch it learn, we'll do passes one at a time using a method called `partial_fit`. Each call to `partial_fit` runs one pass through the training data and then stops — without throwing away what it learned last time. (If you called `fit` twice in a row, it would reset the weights and start over. `partial_fit` does not.)

After each pass we'll record three things on the **test** set:

- **Overall accuracy** (all 10 digits)
- **Precision and recall for digit 1** — an "easy" digit (a single vertical stroke)
- **Precision and recall for digit 8** — a "hard" digit (two loops, easy to confuse with 0, 3, 6, or 9)

We'll also snapshot the network's internal weights at epoch 1 and epoch 20 so we can look at them in Part 2, and snapshot its predicted probabilities for one specific test image so we can look at those in Part 3.

In [ ]:
#@title Train for 20 passes, recording metrics after each — run this cell (~30 seconds)
n_epochs = 20
nn = MLPClassifier(hidden_layer_sizes=(16, 16), random_state=0)
classes = np.arange(10)

epochs      = []
overall_acc = []
prec_1, rec_1 = [], []
prec_8, rec_8 = [], []

# We'll also snapshot weights and probabilities at specific epochs
weights_epoch_1  = None
weights_epoch_20 = None
probas_epoch_1   = None
probas_epoch_5   = None
probas_epoch_20  = None

for epoch in range(1, n_epochs + 1):
    # One pass through the training data
    nn.partial_fit(X_train, y_train, classes=classes)

    pred = nn.predict(X_test)
    overall_acc.append((pred == y_test).mean())

    # precision and recall for digit 1
    pred_1 = (pred == 1)
    actual_1 = (y_test == 1)
    tp1 = (pred_1 & actual_1).sum()
    fp1 = (pred_1 & ~actual_1).sum()
    fn1 = (~pred_1 & actual_1).sum()
    prec_1.append(tp1 / (tp1 + fp1) if (tp1 + fp1) > 0 else 0)
    rec_1.append (tp1 / (tp1 + fn1) if (tp1 + fn1) > 0 else 0)

    # precision and recall for digit 8
    pred_8 = (pred == 8)
    actual_8 = (y_test == 8)
    tp8 = (pred_8 & actual_8).sum()
    fp8 = (pred_8 & ~actual_8).sum()
    fn8 = (~pred_8 & actual_8).sum()
    prec_8.append(tp8 / (tp8 + fp8) if (tp8 + fp8) > 0 else 0)
    rec_8.append (tp8 / (tp8 + fn8) if (tp8 + fn8) > 0 else 0)

    epochs.append(epoch)

    # Snapshots for Parts 2 and 3
    if epoch == 1:
        weights_epoch_1 = nn.coefs_[0].copy()
        probas_epoch_1  = nn.predict_proba(X_test).copy()
    if epoch == 5:
        probas_epoch_5  = nn.predict_proba(X_test).copy()
    if epoch == 20:
        weights_epoch_20 = nn.coefs_[0].copy()
        probas_epoch_20  = nn.predict_proba(X_test).copy()

print("Done.")
print("Final test accuracy:", round(overall_acc[-1], 3))

### Learning curves

In [ ]:
#@title Plot the learning curves — run this cell
plt.figure(figsize=(11, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, overall_acc, marker="o", color="black")
plt.xlabel("training pass (epoch)")
plt.ylabel("test accuracy")
plt.title("Overall test accuracy")
plt.ylim(0, 1)
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(epochs, prec_1, marker="o", label="digit 1 — precision")
plt.plot(epochs, rec_1,  marker="o", label="digit 1 — recall", linestyle="--")
plt.plot(epochs, prec_8, marker="s", label="digit 8 — precision")
plt.plot(epochs, rec_8,  marker="s", label="digit 8 — recall", linestyle="--")
plt.xlabel("training pass (epoch)")
plt.ylabel("score")
plt.title("Easy digit (1) vs hard digit (8)")
plt.ylim(0, 1)
plt.legend(loc="lower right", fontsize=8)
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

**What to notice:**
- Accuracy rises fast at first, then slows down — most of the learning happens in the first few passes.
- The digit 1 curves climb high almost immediately. The digit 8 curves climb more slowly and settle lower. Hard digits take the network longer to figure out — and it still makes more mistakes on them at the end.

---

## Part 2: What the network *is* — a stack of weights

When we say "the network learns," what actually changes? The answer: a big table of numbers called **weights**. The first hidden layer has one weight for each of the 784 input pixels, times 16 neurons — so 784 × 16 = 12,544 numbers in that layer alone. Each neuron has its own set of 784 weights: one weight per pixel, telling that neuron how much to care about each pixel.

Here's the clever trick: because each neuron has exactly 784 weights (one per pixel), we can reshape that set of 784 weights back into a 28×28 image and **look at it**. A bright spot in the image means "this pixel matters a lot to this neuron." A dark spot means "this pixel doesn't matter much."

Let's look at all 16 first-hidden-layer neurons, before the network has had much training (epoch 1) and after 20 passes:

In [ ]:
#@title Weight images: epoch 1 vs epoch 20 — run this cell
plt.figure(figsize=(12, 6))

for i in range(16):
    plt.subplot(4, 8, i + 1)
    plt.imshow(weights_epoch_1[:, i].reshape(28, 28), cmap="seismic",
               vmin=-np.abs(weights_epoch_1).max(), vmax=np.abs(weights_epoch_1).max())
    plt.axis("off")

for i in range(16):
    plt.subplot(4, 8, i + 17)
    plt.imshow(weights_epoch_20[:, i].reshape(28, 28), cmap="seismic",
               vmin=-np.abs(weights_epoch_20).max(), vmax=np.abs(weights_epoch_20).max())
    plt.axis("off")

plt.suptitle("First-hidden-layer weights: epoch 1 (top 2 rows) vs epoch 20 (bottom 2 rows)")
plt.tight_layout()
plt.show()

**What to notice:**
- At epoch 1, the 16 neurons look close to random — the network has barely started rearranging anything.
- By epoch 20, you can see faint structure: blobs, strokes, curves. Each neuron has specialized to notice a different pattern in the pixels. Nobody told the network what to look for — it found these patterns on its own, by being told "right" or "wrong" thousands of times and nudging each weight a little bit after every mistake.

The video's phrase "the weights are what get updated" is exactly this picture — these 12,544 numbers, changed bit by bit across the training passes.

---

## Part 3: Learning from one mistake

We have one more thing snapshotted: the network's predicted probabilities for every test image, at epochs 1, 5, and 20. `predict_proba` gives us the full 10-number output — how confident the network is in each digit — not just the winner.

Let's pick a specific image the network got **wrong at epoch 1** but **right at epoch 20**, and watch its confidence shift across training.

In [ ]:
#@title Find an image the network eventually figures out — run this cell
pred_at_1  = probas_epoch_1.argmax(axis=1)
pred_at_20 = probas_epoch_20.argmax(axis=1)

# Images that were wrong at epoch 1 but right at epoch 20
learned_mask = (pred_at_1 != y_test) & (pred_at_20 == y_test)
learned_indices = np.where(learned_mask)[0]
print(f"{len(learned_indices)} test images were wrong at epoch 1 but right at epoch 20.")

# Pick one to watch
watch = learned_indices[0]
show_digit(X_test[watch], label=y_test[watch], predicted=pred_at_1[watch])
print(f"At epoch 1,  network said: {pred_at_1[watch]}")
print(f"At epoch 20, network said: {pred_at_20[watch]}")

Now plot the network's full 10-way probability output for this *same image* at three points in training:

In [ ]:
#@title Watch the probabilities shift — run this cell
plt.figure(figsize=(12, 3.5))

snapshots = [(1, probas_epoch_1), (5, probas_epoch_5), (20, probas_epoch_20)]

for k in range(3):
    epoch, probas = snapshots[k]
    plt.subplot(1, 3, k + 1)
    colors = ["steelblue"] * 10
    colors[y_test[watch]] = "green"   # true digit in green
    plt.bar(range(10), probas[watch], color=colors)
    plt.xticks(range(10))
    plt.ylim(0, 1)
    plt.xlabel("digit")
    if k == 0:
        plt.ylabel("network's confidence")
    plt.title(f"after {epoch} training pass{'es' if epoch != 1 else ''}")

plt.suptitle(f"Predictions for this one image (true label shown in green)")
plt.tight_layout()
plt.show()

**What to notice:**
- At epoch 1, the network spreads its confidence across several digits — it's not sure.
- By epoch 5, the true digit is often already winning, but the network is still hedging.
- By epoch 20, the true digit's bar is tall and the others are short — the network has become confident and correct.

Nothing about *this image* changed between the three snapshots. What changed is the network — the 12,544 weights you saw in Part 2. The network saw thousands of other training images between epochs, made mistakes on them, and adjusted its weights a little after each one. Those adjustments accumulated until they also happened to work for this image.

That's all "learning" is here: small, repeated corrections to a big pile of numbers, until the numbers are arranged in a way that produces correct answers most of the time. No magic, just a lot of bookkeeping.

---

*Worksheet created by Ethan C. Brown in collaboration with Claude Code.*